In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()


In [2]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia langchain-classic

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━

## Wikipedia Retriever

In [ ]:
from langchain_community.retrievers import WikipediaRetriever

In [ ]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [ ]:
# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

docs = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(docs):
  print(f"\n-----Here is the result of {i+1} document----")
  print(f"Content of the document \n: {doc.page_content}")


-----Here is the result of 1 document----
Content of the document 
: The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the 

## VectorStore Retrievers

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [ ]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [ ]:
# Step 2: Initialize embedding model
embedding_model = OpenAIEmbeddings()

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

In [17]:
retriever = vectorstore.as_retriever(search_kwargs = {"k" : 2})

result = retriever.invoke("Why is Chroma DB used?")

In [18]:
for i, result in enumerate(result):
  print(f"Result is \n {result.page_content}")

Result is 
 Chroma is a vector database optimized for LLM-based search.
Result is 
 LangChain helps developers build LLM applications easily.


In [23]:
results = vectorstore.similarity_search(query, k=2)

In [25]:
for i, doc in enumerate(result):
  print(f"---Result from vector store--- {doc.page_content}")

---Result from vector store--- Chroma is a vector database optimized for LLM-based search.
---Result from vector store--- Embeddings convert text into high-dimensional vectors.


We can do retrieval from the vector stores using Retriver and vector_store.similarity_search() also, but the Retriever gives extra funtionality such as

1. Advance search mechanism
2. Retrievers can be integrated intob Chains

#MMR

In [10]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [5]:
from langchain_community.vectorstores import FAISS

In [13]:
vector_store = FAISS.from_documents(
    documents = docs,
    embedding = OpenAIEmbeddings()
)

In [15]:
# Enable MMR in the retriever
retriever = vector_store.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [17]:
query = "What does Langchain do?"
result = retriever.invoke(query)

In [18]:
for i, doc in enumerate(result):
  print(f"Result no. {i+1}")
  print(f"Content is {doc.page_content}")

Result no. 1
Content is LangChain is used to build LLM based applications.
Result no. 2
Content is Chroma is used to store and search document embeddings.
Result no. 3
Content is LangChain supports Chroma, FAISS, Pinecone, and more.


# Multi Query Retriever

In [31]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers import MultiQueryRetriever

In [32]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [36]:
embedding = OpenAIEmbeddings()
vector_store = FAISS.from_documents(
    documents = all_docs,
    embedding = embedding
)

In [40]:
multi_query_retriever = MultiQueryRetriever.from_llm(retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs ={"k":5}), llm = ChatOpenAI(model = 'gpt-4.1-nano'))

In [42]:
result = multi_query_retriever.invoke("How to improve energy levels and maintain balance?")

In [43]:
for i, doc in enumerate(result):
  print(f"Result no. {i+1}")
  print(f"Result is {doc.page_content}")

Result no. 1
Result is Drinking sufficient water throughout the day helps maintain metabolism and energy.
Result no. 2
Result is Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Result no. 3
Result is Regular walking boosts heart health and can reduce symptoms of depression.
Result no. 4
Result is Deep sleep is crucial for cellular repair and emotional regulation.
Result no. 5
Result is Consuming leafy greens and fruits helps detox the body and improve longevity.


#Contextual Compression Retriever


In [5]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [6]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [9]:
# Create a FAISS vector store from the documents
embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding_model)

In [10]:
base_retriever = vectorstore.as_retriever(search_kwargs = {"k" : 2})

In [11]:
# Set up the compressor using an LLM
llm = ChatOpenAI(model="gpt-4.1-nano")
compressor = LLMChainExtractor.from_llm(llm)

In [12]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [13]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [14]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
